# 03 - Model Inspection

In [ ]:
import osos.chdir(r'C:\Users\avav\Documents\freqtrade')import pandas as pdimport plotly.express as pximport plotly.graph_objects as gofrom pathlib import Pathimport jsonMODELS_DIR = Path(r'C:\Users\avav\Documents\freqtrade\user_data\models\sol_only_v1')

In [ ]:
# List all sub-train metadatasubs = sorted((MODELS_DIR / 'sub-train-SOL_').iterdir() if (MODELS_DIR / 'sub-train-SOL_').exists() else [])subs = [p for p in MODELS_DIR.iterdir() if p.is_dir() and p.name.startswith('sub-train')]print(f"Found {len(subs)} sub-train dirs")for s in subs:    meta = s / (s.name + '_metadata.json')    if meta.exists():        with open(meta) as f:            d = json.load(f)        print(f"  {s.name}: {d.get('training_features_list', [])[:5]}... ({len(d.get('training_features_list', []))} features)")        print(f"    labels: {d.get('label_list', [])}, means: {d.get('labels_mean', {})}, stds: {d.get('labels_std', {})}")

In [ ]:
# Inspect predictions from latest sub-trainpreds_dir = MODELS_DIR / 'backtesting_predictions'preds_files = sorted(preds_dir.glob('*_prediction.feather'))print(f"Found {len(preds_files)} prediction files")# Read all and combineall_preds = []for f in preds_files:    df = pd.read_feather(f)    df['window_start_ts'] = int(f.stem.split('_')[2])    df['window_start'] = pd.to_datetime(df['window_start_ts'], unit='s', utc=True)    all_preds.append(df)combined = pd.concat(all_preds, ignore_index=True)print(f"Combined {len(combined)} predictions")print(combined.columns.tolist())

In [ ]:
# Distribution of predicted classif '&-s_above_median' in combined.columns:    counts = combined['&-s_above_median'].value_counts().sort_index()    print("Class distribution:")    print(counts)    print(f"% predicted 1 (long signal): {counts.get(1, 0) / len(combined) * 100:.1f}%")    fig = px.histogram(combined, x='&-s_above_median', title='Predicted class distribution')    fig.show()

In [ ]:
# Distribution of predicted probabilityprob_cols = [c for c in combined.columns if c.startswith('class_')]print(f"Probability columns: {prob_cols}")for c in prob_cols:    fig = px.histogram(combined, x=c, nbins=50, title=f'Predicted probability: {c}')    fig.show()

In [ ]:
# Confidence vs window start (do predictions change over time?)if 'do_predict' in combined.columns:    dp = combined.groupby('window_start')['do_predict'].mean()    fig = go.Figure()    fig.add_trace(go.Scatter(x=dp.index, y=dp.values, mode='lines+markers', name='do_predict rate'))    fig.update_layout(title='do_predict rate over time (1=confident, 0=NaN)', template='plotly_dark')    fig.show()

In [ ]:
Model inspection template ready.